In [2]:
import pandas as pd
from pathlib import Path

# une todos os CSVs em data/processed/flows e salva em data/processed/flows/combined.csv

src_dir = Path("data_tcc/flows/")
csv_files = sorted(src_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"Nenhum CSV encontrado em {src_dir}")

dfs = [pd.read_csv(f, low_memory=False) for f in csv_files]
combined = pd.concat(dfs, ignore_index=True, sort=False)

out_path = src_dir / "combined.csv"
combined.to_csv(out_path, index=False)
print(f"Salvo {len(combined)} linhas de {len(csv_files)} arquivos em {out_path}")

Salvo 109468 linhas de 5 arquivos em data_tcc\flows\combined.csv


In [3]:
df = pd.read_csv("data_tcc/flows/combined.csv", low_memory=False) 
df.head()

,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,192.168.100.20-192.168.100.10-58210-80-6,192.168.100.20,58210,192.168.100.10,80,6,22/06/2026 02:52:20 AM,2937,5,4,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NeedManualLabel
1,192.168.100.20-192.168.100.10-58226-80-6,192.168.100.20,58226,192.168.100.10,80,6,22/06/2026 02:52:21 AM,4012,6,5,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NeedManualLabel
2,192.168.100.20-192.168.100.10-58240-80-6,192.168.100.20,58240,192.168.100.10,80,6,22/06/2026 02:52:22 AM,3290,5,4,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NeedManualLabel
3,192.168.100.20-192.168.100.10-58244-80-6,192.168.100.20,58244,192.168.100.10,80,6,22/06/2026 02:52:24 AM,3183,6,5,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NeedManualLabel
4,192.168.100.20-192.168.100.10-58256-80-6,192.168.100.20,58256,192.168.100.10,80,6,22/06/2026 02:52:25 AM,6437,6,5,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NeedManualLabel


In [4]:
df2 = pd.read_csv("data_tcc/script_info.csv", low_memory=False) 
df2.head()

,scenario,expected_class,start_time,end_time,command
0,C1 - Trafego benigno HTTP variado,BENIGN,2026-06-22T01:52:19-04:00,2026-06-22T01:52:50-04:00,"curl com caminhos, métodos, cabeçalhos e inter..."
1,C2 - DoS Hulk-like HTTP flood,DoS Hulk,2026-06-22T01:53:20-04:00,2026-06-22T01:53:57-04:00,ApacheBench: ab -n 2000 -c 200 em repetição du...
2,C4 - DoS slowloris,DoS slowloris,2026-06-22T01:54:27-04:00,2026-06-22T01:54:57-04:00,"python slowloris.py URL, limitado pelo tempo d..."
3,C5 - DoS Slowhttptest slow body,DoS Slowhttptest,2026-06-22T01:55:27-04:00,2026-06-22T01:55:49-04:00,slowhttptest -B -c 200 -r 50 -t POST -x 10 -s ...


In [5]:
df2['start_time'] = (
    pd.to_datetime(df2['start_time']) + pd.Timedelta(hours=1)
).dt.strftime('%d/%m/%Y %I:%M:%S %p')

df2['end_time'] = (
    pd.to_datetime(df2['end_time']) + pd.Timedelta(hours=1)
).dt.strftime('%d/%m/%Y %I:%M:%S %p')

df2.head()

,scenario,expected_class,start_time,end_time,command
0,C1 - Trafego benigno HTTP variado,BENIGN,22/06/2026 02:52:19 AM,22/06/2026 02:52:50 AM,"curl com caminhos, métodos, cabeçalhos e inter..."
1,C2 - DoS Hulk-like HTTP flood,DoS Hulk,22/06/2026 02:53:20 AM,22/06/2026 02:53:57 AM,ApacheBench: ab -n 2000 -c 200 em repetição du...
2,C4 - DoS slowloris,DoS slowloris,22/06/2026 02:54:27 AM,22/06/2026 02:54:57 AM,"python slowloris.py URL, limitado pelo tempo d..."
3,C5 - DoS Slowhttptest slow body,DoS Slowhttptest,22/06/2026 02:55:27 AM,22/06/2026 02:55:49 AM,slowhttptest -B -c 200 -r 50 -t POST -x 10 -s ...


In [6]:
df2['start_time_dt'] = pd.to_datetime(
    df2['start_time'],
    format='%d/%m/%Y %I:%M:%S %p'
)

df2['end_time_dt'] = pd.to_datetime(
    df2['end_time'],
    format='%d/%m/%Y %I:%M:%S %p'
)

df2 = df2.sort_values('start_time_dt').reset_index(drop=True)

df2['next_start_time'] = (
    df2['start_time_dt']
    .shift(-1)
    .fillna(df2['end_time_dt'] + pd.Timedelta(seconds=30))
)

In [7]:
df2.head()

,scenario,expected_class,start_time,end_time,command,start_time_dt,end_time_dt,next_start_time
0,C1 - Trafego benigno HTTP variado,BENIGN,22/06/2026 02:52:19 AM,22/06/2026 02:52:50 AM,"curl com caminhos, métodos, cabeçalhos e inter...",2026-06-22 02:52:19,2026-06-22 02:52:50,2026-06-22 02:53:20
1,C2 - DoS Hulk-like HTTP flood,DoS Hulk,22/06/2026 02:53:20 AM,22/06/2026 02:53:57 AM,ApacheBench: ab -n 2000 -c 200 em repetição du...,2026-06-22 02:53:20,2026-06-22 02:53:57,2026-06-22 02:54:27
2,C4 - DoS slowloris,DoS slowloris,22/06/2026 02:54:27 AM,22/06/2026 02:54:57 AM,"python slowloris.py URL, limitado pelo tempo d...",2026-06-22 02:54:27,2026-06-22 02:54:57,2026-06-22 02:55:27
3,C5 - DoS Slowhttptest slow body,DoS Slowhttptest,22/06/2026 02:55:27 AM,22/06/2026 02:55:49 AM,slowhttptest -B -c 200 -r 50 -t POST -x 10 -s ...,2026-06-22 02:55:27,2026-06-22 02:55:49,2026-06-22 02:56:19


In [8]:
def get_real_class(timestamp, df2):
    ts = pd.to_datetime(timestamp, format='%d/%m/%Y %I:%M:%S %p')

    for idx, row in df2.iterrows():
        start = row['start_time_dt']
        next_start = row['next_start_time']
        end = row['end_time_dt']

        # Se não for a última linha, vale até antes do próximo start
        if pd.notna(next_start):
            if start <= ts < next_start:
                return row['expected_class']

        # Se for a última linha, vale até o end_time dela
        else:
            if start <= ts <= end:
                return row['expected_class']

    return 'Unknown'

df['class_real'] = df['Timestamp'].apply(lambda x: get_real_class(x, df2))

In [9]:
df.head()

,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,class_real
0,192.168.100.20-192.168.100.10-58210-80-6,192.168.100.20,58210,192.168.100.10,80,6,22/06/2026 02:52:20 AM,2937,5,4,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NeedManualLabel,BENIGN
1,192.168.100.20-192.168.100.10-58226-80-6,192.168.100.20,58226,192.168.100.10,80,6,22/06/2026 02:52:21 AM,4012,6,5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NeedManualLabel,BENIGN
2,192.168.100.20-192.168.100.10-58240-80-6,192.168.100.20,58240,192.168.100.10,80,6,22/06/2026 02:52:22 AM,3290,5,4,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NeedManualLabel,BENIGN
3,192.168.100.20-192.168.100.10-58244-80-6,192.168.100.20,58244,192.168.100.10,80,6,22/06/2026 02:52:24 AM,3183,6,5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NeedManualLabel,BENIGN
4,192.168.100.20-192.168.100.10-58256-80-6,192.168.100.20,58256,192.168.100.10,80,6,22/06/2026 02:52:25 AM,6437,6,5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NeedManualLabel,BENIGN


In [10]:
df['class_real'].value_counts()


class_real
DoS Hulk            106568
DoS Slowhttptest      1632
DoS slowloris         1204
BENIGN                  64
Name: count, dtype: int64

In [11]:
df.to_csv("data_tcc/flows/combined_with_classes.csv", index=False)

In [12]:
import joblib

model = joblib.load(Path("modelo/random_forest.joblib"))
label_encoder = joblib.load(Path("modelo/label_encoder.joblib"))

print("Modelo e label encoder carregados com sucesso.")

Modelo e label encoder carregados com sucesso.


In [21]:
FEATURE_COLUMN_ALIASES = {
    "Flow Duration": ["Flow Duration"],
    "Total Fwd Packets": ["Total Fwd Packets", "Total Fwd Packet"],
    "Total Backward Packets": ["Total Backward Packets", "Total Bwd packets"],
    "Total Length of Fwd Packets": [
        "Total Length of Fwd Packets",
        "Total Length of Fwd Packet",
    ],
    "Total Length of Bwd Packets": [
        "Total Length of Bwd Packets",
        "Total Length of Bwd Packet",
    ],
    "Fwd Packet Length Max": ["Fwd Packet Length Max"],
    "Fwd Packet Length Min": ["Fwd Packet Length Min"],
    "Fwd Packet Length Mean": ["Fwd Packet Length Mean"],
    "Fwd Packet Length Std": ["Fwd Packet Length Std"],
    "Bwd Packet Length Max": ["Bwd Packet Length Max"],
    "Bwd Packet Length Min": ["Bwd Packet Length Min"],
    "Bwd Packet Length Mean": ["Bwd Packet Length Mean"],
    "Bwd Packet Length Std": ["Bwd Packet Length Std"],
    "Flow Bytes/s": ["Flow Bytes/s"],
    "Flow Packets/s": ["Flow Packets/s"],
    "Flow IAT Mean": ["Flow IAT Mean"],
    "Flow IAT Std": ["Flow IAT Std"],
    "Flow IAT Max": ["Flow IAT Max"],
    "Flow IAT Min": ["Flow IAT Min"],
    "Fwd IAT Total": ["Fwd IAT Total"],
    "Fwd IAT Mean": ["Fwd IAT Mean"],
    "Fwd IAT Std": ["Fwd IAT Std"],
    "Fwd IAT Max": ["Fwd IAT Max"],
    "Fwd IAT Min": ["Fwd IAT Min"],
    "Bwd IAT Total": ["Bwd IAT Total"],
    "Bwd IAT Mean": ["Bwd IAT Mean"],
    "Bwd IAT Std": ["Bwd IAT Std"],
    "Bwd IAT Max": ["Bwd IAT Max"],
    "Bwd IAT Min": ["Bwd IAT Min"],
    "Fwd PSH Flags": ["Fwd PSH Flags"],
    "Fwd Header Length": ["Fwd Header Length"],
    "Bwd Header Length": ["Bwd Header Length"],
    "Fwd Packets/s": ["Fwd Packets/s"],
    "Bwd Packets/s": ["Bwd Packets/s"],
    "Min Packet Length": ["Min Packet Length", "Packet Length Min"],
    "Max Packet Length": ["Max Packet Length", "Packet Length Max"],
    "Packet Length Mean": ["Packet Length Mean"],
    "Packet Length Std": ["Packet Length Std"],
    "Packet Length Variance": ["Packet Length Variance"],
    "FIN Flag Count": ["FIN Flag Count"],
    "SYN Flag Count": ["SYN Flag Count"],
    "PSH Flag Count": ["PSH Flag Count"],
    "ACK Flag Count": ["ACK Flag Count"],
    "Down/Up Ratio": ["Down/Up Ratio"],
    "Average Packet Size": ["Average Packet Size"],
    "Avg Fwd Segment Size": ["Avg Fwd Segment Size", "Fwd Segment Size Avg"],
    "Avg Bwd Segment Size": ["Avg Bwd Segment Size", "Bwd Segment Size Avg"],
    "Subflow Fwd Packets": ["Subflow Fwd Packets"],
    "Subflow Fwd Bytes": ["Subflow Fwd Bytes"],
    "Subflow Bwd Packets": ["Subflow Bwd Packets"],
    "Subflow Bwd Bytes": ["Subflow Bwd Bytes"],
    "Init_Win_bytes_forward": ["Init_Win_bytes_forward", "FWD Init Win Bytes"],
    "Init_Win_bytes_backward": ["Init_Win_bytes_backward", "Bwd Init Win Bytes"],
    "act_data_pkt_fwd": ["act_data_pkt_fwd", "Fwd Act Data Pkts"],
    "min_seg_size_forward": ["min_seg_size_forward", "Fwd Seg Size Min"],
    "Active Mean": ["Active Mean"],
    "Active Std": ["Active Std"],
    "Active Max": ["Active Max"],
    "Active Min": ["Active Min"],
    "Idle Mean": ["Idle Mean"],
    "Idle Std": ["Idle Std"],
    "Idle Max": ["Idle Max"],
    "Idle Min": ["Idle Min"],
}

In [28]:
df_class = pd.read_csv("data_tcc/flows/combined_with_classes.csv", low_memory=False)

In [29]:
import numpy as np

df_class = df_class.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

print(f"df_class limpo: {df_class.shape[0]} linhas, {df_class.shape[1]} colunas")

df_class limpo: 109428 linhas, 85 colunas


In [30]:
df_class.head()

,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,class_real
0,192.168.100.20-192.168.100.10-58210-80-6,192.168.100.20,58210,192.168.100.10,80,6,22/06/2026 02:52:20 AM,2937,5,4,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NeedManualLabel,BENIGN
1,192.168.100.20-192.168.100.10-58226-80-6,192.168.100.20,58226,192.168.100.10,80,6,22/06/2026 02:52:21 AM,4012,6,5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NeedManualLabel,BENIGN
2,192.168.100.20-192.168.100.10-58240-80-6,192.168.100.20,58240,192.168.100.10,80,6,22/06/2026 02:52:22 AM,3290,5,4,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NeedManualLabel,BENIGN
3,192.168.100.20-192.168.100.10-58244-80-6,192.168.100.20,58244,192.168.100.10,80,6,22/06/2026 02:52:24 AM,3183,6,5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NeedManualLabel,BENIGN
4,192.168.100.20-192.168.100.10-58256-80-6,192.168.100.20,58256,192.168.100.10,80,6,22/06/2026 02:52:25 AM,6437,6,5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NeedManualLabel,BENIGN


In [31]:
class_backup = df_class['class_real'].copy()

In [36]:
# Filtrar e renomear colunas de df_class usando FEATURE_COLUMN_ALIASES
found_cols = []
rename_map = {}

for canon, aliases in FEATURE_COLUMN_ALIASES.items():
    for alias in aliases:
        if alias in df_class.columns:
            found_cols.append(alias)
            rename_map[alias] = canon
            break

df_features = df_class[found_cols].rename(columns=rename_map).copy()

print(f"Selected {len(df_features.columns)} features. Shape: {df_features.shape}")
df_features.head()

Selected 63 features. Shape: (109428, 63)


,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,...,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
0,2937,5,4,113.0,255.0,113.0,0.0,22.600000,50.535136,255.0,...,1,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,4012,6,5,98.0,10926.0,98.0,0.0,16.333333,40.008332,7240.0,...,1,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3290,5,4,96.0,255.0,96.0,0.0,19.200000,42.932505,255.0,...,1,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3183,6,5,101.0,10926.0,101.0,0.0,16.833333,41.233077,7240.0,...,1,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,6437,6,5,111.0,10926.0,111.0,0.0,18.500000,45.315560,7240.0,...,1,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [38]:
# alinhar colunas com as features esperadas pelo modelo
X = df_features.copy()

if "Fwd Header Length.1" in X.columns and "Fwd Header Length" not in X.columns:
    X = X.rename(columns={"Fwd Header Length.1": "Fwd Header Length"})

missing = [c for c in expected_features if c not in X.columns]
if missing:
    raise ValueError(f"Features ausentes: {missing}")

X = X[expected_features]

pred_encoded = model.predict(X)
predicted_class = label_encoder.inverse_transform(pred_encoded)

df_features["predicted_class"] = predicted_class

df_features[["predicted_class"]].head()

,predicted_class
0,BENIGN
1,BENIGN
2,BENIGN
3,BENIGN
4,BENIGN


In [39]:
df_features

,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,predicted_class
0,2937,5,4,113.0,255.0,113.0,0.0,22.600000,50.535136,255.0,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
1,4012,6,5,98.0,10926.0,98.0,0.0,16.333333,40.008332,7240.0,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
2,3290,5,4,96.0,255.0,96.0,0.0,19.200000,42.932505,255.0,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
3,3183,6,5,101.0,10926.0,101.0,0.0,16.833333,41.233077,7240.0,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
4,6437,6,5,111.0,10926.0,111.0,0.0,18.500000,45.315560,7240.0,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109423,528,2,1,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
109424,7231,2,1,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
109425,1320,2,1,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
109426,65,2,1,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN


In [40]:
df_features.shape, df_class.shape

((109428, 64), (109428, 85))

In [41]:
df_features['class_real'] = class_backup

In [44]:
df_features.tail()

,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,predicted_class,class_real
109423,528,2,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN,DoS Slowhttptest
109424,7231,2,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN,DoS Slowhttptest
109425,1320,2,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN,DoS Slowhttptest
109426,65,2,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN,DoS Slowhttptest
109427,280,2,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN,DoS Slowhttptest


In [45]:
df_features.to_csv("data_tcc/flows/predicted_classes.csv", index=False)